# Docs 7 — Store and Ask

Write the table where others can use it, then make questions cheap — with
code doing the arithmetic and the LLM only at the edges.

In [ ]:
validated = [
 {"month":"2026-01","families":167,"donations":1210.00},
 {"month":"2026-02","families":174,"donations":1385.50},
 {"month":"2026-03","families":212,"donations":1847.50},
 {"month":"2026-04","families":198,"donations":2210.00},
 {"month":"2026-05","families":241,"donations":1655.25},
 {"month":"2026-06","families":188,"donations":1500.00},
 {"month":"2026-07","families":176,"donations":980.00},
 {"month":"2026-08","families":205,"donations":1344.75},
 {"month":"2026-09","families":189,"donations":1760.10},
 {"month":"2026-10","families":214,"donations":1922.40},
 {"month":"2026-11","families":232,"donations":2455.00},
 {"month":"2026-12","families":251,"donations":3010.75},
]

In [ ]:
import csv, json

with open("pantry_2026.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["month", "families", "donations"])
    w.writeheader()
    w.writerows(validated)

with open("pantry_2026.json", "w") as f:
    json.dump(validated, f, indent=1)

print(open("pantry_2026.csv").read())
print("Two files. The CSV opens in Excel/Sheets - the organization can use")
print("it without you. That property is most of the point of storing well.")

## Questions are one line now — and they show their work

In [ ]:
spring = [r for r in validated if "2026-03" <= r["month"] <= "2026-05"]
print(f"Spring donations: ${sum(r['donations'] for r in spring):,.2f}"
      f"  (rows: {[r['month'] for r in spring]})")

busiest = max(validated, key=lambda r: r["families"])
print(f"Busiest month: {busiest['month']} with {busiest['families']} families")

avg = sum(r["families"] for r in validated) / len(validated)
dips = [r["month"] for r in validated if r["families"] < avg * 0.92]
print(f"Months well under average: {dips}")

## The LLM at the edges — and only the edges

Turning a vague stakeholder question into the right filter, and a computed
number into a sentence. **The number itself always comes from the code
above.** (Precomputed responses shown, so this section reads fine without
a key.)

In [ ]:
%pip install -q anthropic
import os, anthropic
from getpass import getpass
os.environ.setdefault('ANTHROPIC_API_KEY', getpass('Class API key: '))
MODEL = 'claude-opus-5'
client = anthropic.Anthropic()
def ask(prompt, max_tokens=300):
    r = client.messages.create(model=MODEL, max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}])
    return ''.join(b.text for b in r.content if b.type == 'text')

# Edge 1: vague question -> concrete filter (the model proposes, YOU approve).
q = "How did we do over the holidays?"
print(ask(f"A pantry table has columns month, families, donations for 2026. "
          f"A board member asks: '{q}'. Propose the exact month range to "
          f"filter (just the range and one sentence why)."))
# Precomputed: "2026-11 through 2026-12 - 'the holidays' most naturally
#  means November-December giving season."

# Edge 2: computed number -> newsletter sentence.
total = sum(r["donations"] for r in validated if r["month"] >= "2026-11")
print(ask(f"Write one warm newsletter sentence reporting that holiday-season "
          f"donations (Nov-Dec) totaled ${total:,.2f}. Do not change the number."))
# Precomputed: "Thanks to your generosity, our November and December drives
#  brought in $5,465.75 to keep neighborhood tables full this holiday season."

Note the division of labor in both edges: the model proposed a filter
— you approved it; the model phrased a sentence — the number came from your
line of code. The moment a number comes from anywhere but the table, you've
built a rumor machine with good manners.

## Turn-in

Four questions of your own answered in code (each naming its rows), and one
newsletter paragraph where every number traces to a line above it.